# DescriPyTor — Modeling

This notebook demonstrates regression and classification model search on the example datasets in `modeling_example/`.

The feature CSVs here were produced by the Features notebook (or the CLI extractor).
The `output` column is the regression target; `class` is the classification target.

**Workflow:**
1. Set up imports and ROOT_DIR
2. Load the example feature CSV
3. Run `search_models` to find the best feature combinations
4. Inspect top models and plot a selected combination

In [ ]:
# ===Jupyter notebook setup===
%reload_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# ROOT_DIR is the MolFeatures directory (one level above this notebook)
ROOT_DIR = str(Path().resolve().parent)
print(f'ROOT_DIR: {ROOT_DIR}')

sys.path.append(ROOT_DIR)
sys.path.append(os.path.join(ROOT_DIR, 'M3_modeler'))
sys.path.append(os.path.join(ROOT_DIR, 'M2_data_extractor'))
sys.path.append(os.path.join(ROOT_DIR, 'utils'))
os.chdir(ROOT_DIR)

for _mod in ['modeling', 'plot', 'help_functions']:
    sys.modules.pop(_mod, None)

try:
    from modeling import LinearRegressionModel, ClassificationModel
    import plot
    print('Imports OK')
except ModuleNotFoundError as e:
    print(f'Import error: {e}\nCheck that ROOT_DIR points to the MolFeatures directory.')

pd.set_option('display.max_columns', None)

NOTEBOOK_DIR = Path(ROOT_DIR) / 'Getting_started_with_examples'
MODELING_DIR = NOTEBOOK_DIR / 'modeling_example'

In [ ]:
# === Regression example ===
# Linear_Dataset_Example.csv has features + an 'output' column as the regression target.

csv_path = str(MODELING_DIR / 'Linear_Dataset_Example.csv')
df = pd.read_csv(csv_path)
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Build the regression model and search over feature combinations

csv_filepaths = {
    'features_csv_filepath': csv_path,
    'target_csv_filepath': ''
}

regression_model = LinearRegressionModel(
    csv_filepaths,
    process_method='one csv',
    y_value='output',
    leave_out=[],          # list of sample names/indices to hold out as external validation
    min_features_num=1,
    max_features_num=3,
)

# Search for the best feature combinations (returns a ranked DataFrame)
results = regression_model.search_models(top_n=10, threshold=0.5)
results

In [ ]:
# === Plot the top model ===
# Replace the feature list below with the best combination from the results table above.

top_features = list(results.iloc[0]['features'])  # auto-pick top combo
print(f'Plotting: {top_features}')
plot.run_single_combo_report(regression_model, features=top_features)

In [ ]:
# === Classification example ===
# Logistic_Dataset_Example.csv has features + a 'class' column as the binary target.

cls_csv_path = str(MODELING_DIR / 'Logistic_Dataset_Example.csv')
df_cls = pd.read_csv(cls_csv_path)
print(f'Classification dataset shape: {df_cls.shape}')
print(f'Class distribution:\n{df_cls["class"].value_counts()}')

cls_filepaths = {
    'features_csv_filepath': cls_csv_path,
    'target_csv_filepath': ''
}

classification_model = ClassificationModel(
    cls_filepaths,
    process_method='one csv',
    y_value='class',
    leave_out=[],
    min_features_num=1,
    max_features_num=2,
)

cls_results = classification_model.search_models(top_n=10, threshold=0.2)
cls_results